# Experiment 1: Feature Selection Comparison

This experiment compares No RFE (all 68 encoded features), RFECV (feature count selected using three-fold cross validation and F1 score), and the RFE20 baseline (20 selected features). Logistic Regression ranks features for RFE20 and RFECV. Logistic Regression, Gradient Boosting and Random Forest are compared using five-fold validation, MICE, one-hot encoding and SMOTE.


## Load and inspect the data

Import the libraries and load the saved training/test split.


In [1]:
import pandas as pd
import joblib
import numpy as np


In [2]:
dt = joblib.load('phase3_data/start_data.joblib')


In [3]:
x_train = dt['x_train']
x_test = dt['x_test']
y_train = dt['y_train']
y_test = dt['y_test']


In [4]:
x_train.shape


(4507, 23)

In [5]:
x_test.shape


(1127, 23)

In [6]:
y_train.shape


(4507,)

In [7]:
y_test.shape


(1127,)

In [8]:
y_train.value_counts()


diabetes
0.0    3685
1.0     822
Name: count, dtype: int64

In [9]:
y_test.value_counts()


diabetes
0.0    922
1.0    205
Name: count, dtype: int64

## Prepare the validation folds

Load the saved MICE-imputed training and validation data for each fold.

In [10]:
from pathlib import Path

def apply_mice(train_data, validation_data, folder_name):
    folder = Path(folder_name)
    frames = []
    for filename, original in [('train_imputed.csv', train_data),('validation_imputed.csv', validation_data)]:
        path = folder / filename
        if not path.exists():
            raise FileNotFoundError(f'Missing {path}')
        frame = pd.read_csv(path)
        if frame.isna().any().any():
            raise ValueError(f'Missing values in baseline MICE data: {path}')
        frame.index = original.index
        frames.append(frame)
    return tuple(frames)


In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from pathlib import Path


### Five-fold validation and one-hot encoding



In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
encoded_folds = []
fold_num = 1

for train_idx, val_idx in cv.split(x_train, y_train):

    print("Preparing fold", fold_num, "/5")
    x_train_fold = x_train.iloc[train_idx]
    x_val_fold = x_train.iloc[val_idx]
    y_train_fold = y_train.iloc[train_idx]
    y_val_fold = y_train.iloc[val_idx]


    cache_path = f"mice_cache/fold_{fold_num}"
    x_train_fold, x_val_fold = apply_mice(x_train_fold,x_val_fold, cache_path)

    encod = OneHotEncoder(handle_unknown="ignore", drop=None,sparse_output=False)
    enco_x_train_fold = encod.fit_transform(x_train_fold.astype(str))
    enco_x_val_fold = encod.transform(x_val_fold.astype(str))

    print("Training shape after OneHotEncoder:", enco_x_train_fold.shape)
    print("Validation shape after OneHotEncoder:", enco_x_val_fold.shape)


    fold_data = {"fold": fold_num,"encoder": encod,"x_train": enco_x_train_fold, "x_val": enco_x_val_fold,"y_train": y_train_fold.to_numpy(),
        "y_val": y_val_fold.to_numpy(),"feature_names": encod.get_feature_names_out()}

    encoded_folds.append(fold_data)
    fold_num += 1

print("\nAll five encoded folds are ready.")


Preparing fold 1 /5
Training shape after OneHotEncoder: (3605, 68)
Validation shape after OneHotEncoder: (902, 68)
Preparing fold 2 /5
Training shape after OneHotEncoder: (3605, 68)
Validation shape after OneHotEncoder: (902, 68)
Preparing fold 3 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Preparing fold 4 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)
Preparing fold 5 /5
Training shape after OneHotEncoder: (3606, 68)
Validation shape after OneHotEncoder: (901, 68)

All five encoded folds are ready.


In [14]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))


'12'

## Feature selection

### No RFE

Keep all encoded features and apply SMOTE only to the training folds.


In [15]:
no_rfe_folds = []
for fold_num, fold in enumerate(encoded_folds, start=1):
    smote = SMOTE(random_state=42)
    x_balan , y_balan = smote.fit_resample(fold['x_train'],fold['y_train'])
    no_rfe_folds.append({ 'fold': fold_num, 'x_train':x_balan , 'y_train': y_balan,'x_val' : fold['x_val']
        , 'y_val' : fold['y_val'], 'features' : fold['x_train'].shape[1]})

print("No RFE ready:", len(no_rfe_folds),"folds")


c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 546, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\chali\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1022, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  

No RFE ready: 5 folds


### RFECV

Use Logistic Regression as the ranking estimator and three-fold cross validation with F1 scoring to select the number of features.


In [16]:
from sklearn.feature_selection import RFECV


In [ ]:
# Create the results folder
OUTPUT = Path("results/untitled180")
OUTPUT.mkdir(parents=True, exist_ok=True)
rfe_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


In [ ]:
# Store the prepared folds and the feature names selected in each fold.
rfecv_folds = []
selected_feature_rows = []

for fold_num, fold in enumerate(encoded_folds, start=1):
    print('RFECV fold', fold_num)

    selector_model = LogisticRegression(solver='liblinear',max_iter=1000,random_state=42)
    selector = RFECV(estimator=selector_model,step=1,cv=rfe_cv,scoring='f1',min_features_to_select=1, n_jobs=1)

    selected_train = selector.fit_transform(fold['x_train'],fold['y_train'])
    selected_val = selector.transform(fold['x_val'])
    print('Selected features:', selector.n_features_)

    for feature_name in fold['feature_names'][selector.support_]:
        selected_feature_rows.append({'Fold': fold_num,'Feature': feature_name })


    smote = SMOTE(random_state=42)
    balanced_x, balanced_y = smote.fit_resample(selected_train,fold['y_train'] )
    rfecv_folds.append({'fold': fold_num,'x_train': balanced_x,'y_train': balanced_y,'x_val': selected_val, 'y_val': fold['y_val'],'selector': selector,'features': selector.n_features_ })


pd.DataFrame(selected_feature_rows).to_csv( OUTPUT / 'rfecv_selected_features.csv', index=False)
print('RFECV folds ready:', len(rfecv_folds))


RFECV fold 1
Selected features: 4
RFECV fold 2
Selected features: 6
RFECV fold 3
Selected features: 48
RFECV fold 4
Selected features: 32
RFECV fold 5
Selected features: 31
RFECV folds ready: 5


## Model Training


## 1. Logistic Regression




### Logistic Regression without RFE


In [19]:
lr_no_rfe_models = []
lr_no_rfe_predictions = []
lr_no_rfe_probabilities = []

for fold_num, fold in enumerate(no_rfe_folds, start=1):
    print("logistic_regression_no_rfe: training fold", fold_num, "/", len(no_rfe_folds))

    model = LogisticRegression(penalty='l2', C=0.1, solver='saga', random_state=42)
    model.fit(fold["x_train"], fold["y_train"])

    y_pred = model.predict(fold["x_val"])
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    lr_no_rfe_predictions.append(y_pred)
    lr_no_rfe_probabilities.append(y_prob)

    lr_no_rfe_models.append({
        "fold": fold_num,
        "model": model,
        "encoder": encoded_folds[fold_num - 1]["encoder"],
        "selector": None,
        "input_columns": list(x_train.columns) })


logistic_regression_no_rfe: training fold 1 / 5
logistic_regression_no_rfe: training fold 2 / 5
logistic_regression_no_rfe: training fold 3 / 5
logistic_regression_no_rfe: training fold 4 / 5
logistic_regression_no_rfe: training fold 5 / 5


### Evaluate Logistic Regression without RFE

Calculate accuracy, precision, recall, F1, ROC-AUC, PR-AUC, balanced accuracy and specificity for each validation fold.


In [20]:
from sklearn.metrics import ( accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,roc_auc_score, precision_recall_curve, auc)
from IPython.display import display

metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy", "specificity"]
lr_no_rfe_metrics = pd.DataFrame(columns=["fold", "features"] + metric_columns)

for i, fold in enumerate(no_rfe_folds):
    y_true = fold["y_val"]
    y_pred = lr_no_rfe_predictions[i]
    y_prob = lr_no_rfe_probabilities[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    lr_fold_acc = accuracy_score(y_true, y_pred)
    lr_fold_precision = precision_score(y_true, y_pred)
    lr_fold_recall = recall_score(y_true, y_pred)
    lr_fold_f1 = f1_score(y_true, y_pred)
    lr_fold_roc_auc = roc_auc_score(y_true, y_prob)
    lr_fold_pr_auc = auc(recall, precision)

    lr_fold_balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    lr_fold_specificity = recall_score(y_true, y_pred, pos_label=0)

    lr_no_rfe_metrics.loc[i] = [
        i + 1, fold["features"],
        lr_fold_acc, lr_fold_precision, lr_fold_recall, lr_fold_f1, lr_fold_roc_auc, lr_fold_pr_auc, lr_fold_balanced_accuracy, lr_fold_specificity]

lr_balanced_accuracy = lr_no_rfe_metrics["balanced_accuracy"].mean()
lr_specificity = lr_no_rfe_metrics["specificity"].mean()


Calculate the mean validation scores across the five folds.

In [21]:
lr_acc = lr_no_rfe_metrics["accuracy"].mean()
lr_precision = lr_no_rfe_metrics["precision"].mean()
lr_recall = lr_no_rfe_metrics["recall"].mean()
lr_f1 = lr_no_rfe_metrics["f1"].mean()
lr_roc_auc = lr_no_rfe_metrics["roc_auc"].mean()
lr_pr_auc = lr_no_rfe_metrics["pr_auc"].mean()


In [22]:
lr_no_rfe_summary = lr_no_rfe_metrics[metric_columns].agg(["mean", "std"]).T


In [23]:
display(lr_no_rfe_metrics)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,68.0,0.725055,0.369085,0.709091,0.485477,0.791143,0.410629,0.718860,0.728630
1,2.0,68.0,0.715078,0.361446,0.727273,0.482897,0.766120,0.433167,0.719810,0.712347
2,3.0,68.0,0.724750,0.370370,0.731707,0.491803,0.809834,0.512171,0.727455,0.723202
3,4.0,68.0,0.702553,0.343373,0.695122,0.459677,0.775805,0.484339,0.699664,0.704206
4,5.0,68.0,0.712542,0.351097,0.682927,0.463768,0.781737,0.480919,0.701029,0.719132


In [24]:
display(lr_no_rfe_summary)


,mean,std
accuracy,0.715996,0.009384
precision,0.359074,0.011658
recall,0.709224,0.020747
f1,0.476725,0.014147
roc_auc,0.784928,0.016629
pr_auc,0.464245,0.041289
balanced_accuracy,0.713364,0.012351
specificity,0.717503,0.009517


### Logistic Regression with RFECV


In [ ]:
lr_rfecv_models = []
lr_rfecv_predictions = []
lr_rfecv_probabilities = []

for fold_num, fold in enumerate(rfecv_folds, start=1):
    print("logistic_regression_rfecv: training fold", fold_num, "/", len(rfecv_folds))
    model = LogisticRegression(penalty='l2', C=0.1, solver='saga', random_state=42)
    model.fit(fold["x_train"], fold["y_train"])  
    y_pred = model.predict(fold["x_val"])
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    lr_rfecv_predictions.append(y_pred)
    lr_rfecv_probabilities.append(y_prob)

    lr_rfecv_models.append({"fold": fold_num,"model": model,"encoder": encoded_folds[fold_num - 1]["encoder"],
        "selector": fold["selector"],"input_columns": list(x_train.columns)})


logistic_regression_rfecv: training fold 1 / 5
logistic_regression_rfecv: training fold 2 / 5
logistic_regression_rfecv: training fold 3 / 5
logistic_regression_rfecv: training fold 4 / 5
logistic_regression_rfecv: training fold 5 / 5


### Evaluate Logistic Regression with RFECV

Calculate the same validation metrics used for the No RFE variant.


In [26]:
lr_rfecv_metrics = pd.DataFrame(columns=["fold", "features"] + metric_columns)

for i, fold in enumerate(rfecv_folds):
    y_true = fold["y_val"]
    y_pred = lr_rfecv_predictions[i]
    y_prob = lr_rfecv_probabilities[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    lr_rfecv_fold_acc = accuracy_score(y_true, y_pred)
    lr_rfecv_fold_precision = precision_score(y_true, y_pred)
    lr_rfecv_fold_recall = recall_score(y_true, y_pred)
    lr_rfecv_fold_f1 = f1_score(y_true, y_pred)
    lr_rfecv_fold_roc_auc = roc_auc_score(y_true, y_prob)
    lr_rfecv_fold_pr_auc = auc(recall, precision)

    lr_rfecv_fold_balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    lr_rfecv_fold_specificity = recall_score(y_true, y_pred, pos_label=0)

    lr_rfecv_metrics.loc[i] = [i + 1, fold["features"],
    lr_rfecv_fold_acc, lr_rfecv_fold_precision, lr_rfecv_fold_recall, lr_rfecv_fold_f1, lr_rfecv_fold_roc_auc, lr_rfecv_fold_pr_auc, lr_rfecv_fold_balanced_accuracy, lr_rfecv_fold_specificity]

lr_rfecv_balanced_accuracy = lr_rfecv_metrics["balanced_accuracy"].mean()
lr_rfecv_specificity = lr_rfecv_metrics["specificity"].mean()


Calculate the mean and standard deviation of the validation metrics across the five folds.


In [27]:
lr_rfecv_acc = lr_rfecv_metrics["accuracy"].mean()
lr_rfecv_precision = lr_rfecv_metrics["precision"].mean()
lr_rfecv_recall = lr_rfecv_metrics["recall"].mean()
lr_rfecv_f1 = lr_rfecv_metrics["f1"].mean()
lr_rfecv_roc_auc = lr_rfecv_metrics["roc_auc"].mean()
lr_rfecv_pr_auc = lr_rfecv_metrics["pr_auc"].mean()

lr_rfecv_summary = lr_rfecv_metrics[metric_columns].agg(["mean", "std"]).T


In [28]:
display(lr_rfecv_metrics)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,4.0,0.723947,0.360000,0.654545,0.464516,0.767230,0.450456,0.697015,0.739484
1,2.0,6.0,0.727273,0.353791,0.593939,0.443439,0.747506,0.441112,0.675531,0.757123
2,3.0,48.0,0.730300,0.375394,0.725610,0.494802,0.808998,0.508262,0.728477,0.731343
3,4.0,32.0,0.706992,0.347561,0.695122,0.463415,0.774332,0.475778,0.702378,0.709634
4,5.0,31.0,0.715871,0.358025,0.707317,0.475410,0.783570,0.485981,0.712546,0.717775


In [29]:
display(lr_rfecv_summary)


,mean,std
accuracy,0.720877,0.009447
precision,0.358954,0.010353
recall,0.675307,0.052439
f1,0.468316,0.018770
roc_auc,0.776327,0.022568
pr_auc,0.472318,0.027122
balanced_accuracy,0.703189,0.019562
specificity,0.731072,0.018614


### Save the Logistic Regression models

In [30]:
MODEL_DIR = OUTPUT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(".............saving the lr models.............")
joblib.dump({ "variant": "logistic_regression_no_rfe","positive_class": 1,"requires_mice_imputed_input": True, "fold_models": lr_no_rfe_models,},MODEL_DIR / "logistic_regression_no_rfe.joblib")
print("Saved logistic regression with no RFE model")

joblib.dump({"variant": "logistic_regression_rfecv", "positive_class": 1,"requires_mice_imputed_input": True, "fold_models": lr_rfecv_models,}, MODEL_DIR / "logistic_regression_rfecv.joblib")
print("Saved logistic_regression model with RFE")
print("..............all the models are done saving...............")


.............saving the lr models.............
Saved logistic regression with no RFE model
Saved logistic_regression model with RFE
..............all the models are done saving...............


## 2. Gradient Boosting Classifier model

In [31]:
from sklearn.ensemble import GradientBoostingClassifier


### Gradient boosting: No RFE


In [32]:
gb_no_rfe_models = []
gb_norfe_predictions = []
gb_norfe_probabilities = []

for fold_no, fold in enumerate(no_rfe_folds, start=1):
    print("gradient_boosting_no_rfe: training fold", fold_no, "/", len(no_rfe_folds))
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=1.0, min_samples_split=2, random_state=42)
    model.fit(fold["x_train"], fold["y_train"])

    y_pre = model.predict(fold["x_val"])
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    gb_norfe_predictions.append(y_pre)
    gb_norfe_probabilities.append(y_prob)

    gb_no_rfe_models.append({"fold": fold_no, "model": model,"encoder": encoded_folds[fold_no - 1]["encoder"],"selector": None,"input_columns": list(x_train.columns)})


gradient_boosting_no_rfe: training fold 1 / 5
gradient_boosting_no_rfe: training fold 2 / 5
gradient_boosting_no_rfe: training fold 3 / 5
gradient_boosting_no_rfe: training fold 4 / 5
gradient_boosting_no_rfe: training fold 5 / 5


calculating the metrics

In [33]:
gb_no_rfe_metrics = pd.DataFrame(columns=["fold", "features"] + metric_columns)

for i, fold in enumerate(no_rfe_folds):
    y_true = fold["y_val"]
    y_pre = gb_norfe_predictions[i]
    y_prob = gb_norfe_probabilities[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    gb_fold_acc = accuracy_score(y_true, y_pre)
    gb_fold_precision = precision_score(y_true, y_pre)
    gb_fold_recall = recall_score(y_true, y_pre)
    gb_fold_f1 = f1_score(y_true, y_pre)
    gb_fold_roc_auc = roc_auc_score(y_true, y_prob)
    gb_fold_pr_auc = auc(recall, precision)
    gb_fold_balanced_accuracy = balanced_accuracy_score(y_true, y_pre)
    gb_fold_specificity = recall_score(y_true, y_pre, pos_label=0)

    gb_no_rfe_metrics.loc[i] = [i + 1, fold["features"],gb_fold_acc, gb_fold_precision, gb_fold_recall, gb_fold_f1, gb_fold_roc_auc, gb_fold_pr_auc, gb_fold_balanced_accuracy, gb_fold_specificity]

gb_balanced_accuracy = gb_no_rfe_metrics["balanced_accuracy"].mean()
gb_specificity = gb_no_rfe_metrics["specificity"].mean()



gb_acc = gb_no_rfe_metrics["accuracy"].mean()
gb_precision = gb_no_rfe_metrics["precision"].mean()
gb_recall = gb_no_rfe_metrics["recall"].mean()
gb_f1 = gb_no_rfe_metrics["f1"].mean()
gb_roc_auc = gb_no_rfe_metrics["roc_auc"].mean()
gb_pr_auc = gb_no_rfe_metrics["pr_auc"].mean()


In [34]:
gb_no_rfe_summary = gb_no_rfe_metrics[metric_columns].agg(["mean", "std"]).T


In [35]:
display(gb_no_rfe_metrics)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,68.0,0.805987,0.466216,0.418182,0.440895,0.791308,0.414696,0.655495,0.892809
1,2.0,68.0,0.803769,0.459459,0.412121,0.434505,0.770051,0.434699,0.651787,0.891452
2,3.0,68.0,0.815760,0.493506,0.463415,0.477987,0.808502,0.523704,0.678790,0.894166
3,4.0,68.0,0.807991,0.470199,0.432927,0.450794,0.777087,0.472020,0.662189,0.891452
4,5.0,68.0,0.810211,0.478528,0.475610,0.477064,0.788525,0.468967,0.680139,0.884668


In [36]:
display(gb_no_rfe_summary)


,mean,std
accuracy,0.808744,0.004591
precision,0.473582,0.013101
recall,0.440451,0.027922
f1,0.456249,0.020274
roc_auc,0.787095,0.014737
pr_auc,0.462817,0.041641
balanced_accuracy,0.665680,0.013133
specificity,0.890909,0.003666


In [37]:
joblib.dump({"variant": "gradient_boosting_no_rfe","positive_class": 1,"requires_mice_imputed_input": True, "fold_models": gb_no_rfe_models,}, MODEL_DIR / "gradient_boosting_no_rfe.joblib")
print("Saved gradient_boosting_no_rfe.joblib")


Saved gradient_boosting_no_rfe.joblib


### Gradient boosting: RFECV


In [ ]:
gb_rfecv_models = []
gb_rfecv_proba = []
gb_rfecv_pred = []


for no_fold, fold in enumerate(rfecv_folds, start=1):
    print("gradient_boosting_rfecv: training fold", no_fold, "/", len(rfecv_folds))
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=1.0, min_samples_split=2, random_state=42)
    model.fit(fold["x_train"], fold["y_train"])
    y_pred = model.predict(fold["x_val"])
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    gb_rfecv_pred.append(y_pred)
    gb_rfecv_proba.append(y_prob)

    gb_rfecv_models.append({"fold": no_fold,"model": model,"encoder": encoded_folds[no_fold - 1]["encoder"],
        "selector": fold["selector"],"input_columns": list(x_train.columns)})


gradient_boosting_rfecv: training fold 1 / 5
gradient_boosting_rfecv: training fold 2 / 5
gradient_boosting_rfecv: training fold 3 / 5
gradient_boosting_rfecv: training fold 4 / 5
gradient_boosting_rfecv: training fold 5 / 5


model evaluation

In [39]:
gb_rfecv_metrics = pd.DataFrame(columns=["fold", "features"] + metric_columns)

for i, fold in enumerate(rfecv_folds):
    y_true = fold["y_val"]
    y_pred = gb_rfecv_pred[i]
    y_prob = gb_rfecv_proba[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    rfecv_fold_acc = accuracy_score(y_true, y_pred)
    rfecv_fold_precision = precision_score(y_true, y_pred, zero_division=0)
    rfecv_fold_recall = recall_score(y_true, y_pred)
    rfecv_fold_f1 = f1_score(y_true, y_pred)
    rfecv_gb_fold_roc_auc = roc_auc_score(y_true, y_prob)
    rfecv_gb_fold_pr_auc = auc(recall, precision)

    gb_rfecv_fold_balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    gb_rfecv_fold_specificity = recall_score(y_true, y_pred, pos_label=0)

    gb_rfecv_metrics.loc[i] = [
        i + 1, fold["features"],rfecv_fold_acc, rfecv_fold_precision, rfecv_fold_recall, rfecv_fold_f1,rfecv_gb_fold_roc_auc, rfecv_gb_fold_pr_auc, gb_rfecv_fold_balanced_accuracy, gb_rfecv_fold_specificity]

gb_rfecv_balanced_accuracy = gb_rfecv_metrics["balanced_accuracy"].mean()
gb_rfecv_specificity = gb_rfecv_metrics["specificity"].mean()



In [40]:
gb_rfecv_acc = gb_rfecv_metrics["accuracy"].mean()
gb_rfecv_precision = gb_rfecv_metrics["precision"].mean()
gb_rfecv_recall = gb_rfecv_metrics["recall"].mean()
gb_rfecv_f1 = gb_rfecv_metrics["f1"].mean()
gb_rfecv_roc_auc = gb_rfecv_metrics["roc_auc"].mean()
gb_rfecv_pr_auc = gb_rfecv_metrics["pr_auc"].mean()

gb_rfecv_summary = gb_rfecv_metrics[metric_columns].agg(["mean", "std"]).T


In [41]:
display(gb_rfecv_metrics)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,4.0,0.651885,0.323877,0.830303,0.465986,0.769656,0.453882,0.721122,0.611940
1,2.0,6.0,0.662971,0.318538,0.739394,0.445255,0.746820,0.438398,0.692628,0.645862
2,3.0,48.0,0.809101,0.476744,0.500000,0.488095,0.803418,0.507280,0.688942,0.877883
3,4.0,32.0,0.779134,0.416268,0.530488,0.466488,0.770543,0.468241,0.682476,0.834464
4,5.0,31.0,0.788013,0.427807,0.487805,0.455840,0.783619,0.447883,0.671311,0.854817


In [42]:
display(gb_rfecv_summary)


,mean,std
accuracy,0.738221,0.074655
precision,0.392647,0.069080
recall,0.617598,0.156797
f1,0.464333,0.015875
roc_auc,0.774811,0.020762
pr_auc,0.463137,0.026949
balanced_accuracy,0.691296,0.018534
specificity,0.764993,0.125754


In [43]:
joblib.dump({"variant": "gradient_boosting_rfecv","positive_class": 1, "requires_mice_imputed_input": True, "fold_models": gb_rfecv_models,}, MODEL_DIR / "gradient_boosting_rfecv.joblib")
print("Saved gradient_boosting_rfecv.joblib")


Saved gradient_boosting_rfecv.joblib


## 3. Random Forest Classifier model

In [44]:
from sklearn.ensemble import  RandomForestClassifier


### Random forest: No RFE


In [45]:
rf_num_models = []
rf_num_predictions = []
rf_num_probabilities = []

for fold_num, fold in enumerate(no_rfe_folds, start=1):
    print("random_forest_no_rfe: training fold", fold_num, "/", len(no_rfe_folds))
    model = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=2, max_features='log2', random_state=42)
    model.fit(fold["x_train"], fold["y_train"])
    y_pred = model.predict(fold["x_val"])
    y_prob = model.predict_proba(fold["x_val"])[:, 1]
    rf_num_predictions.append(y_pred)
    rf_num_probabilities.append(y_prob)
    rf_num_models.append({ "fold": fold_num,"model": model,"encoder": encoded_folds[fold_num - 1]["encoder"], "selector": None,
                          "input_columns": list(x_train.columns)})


random_forest_no_rfe: training fold 1 / 5
random_forest_no_rfe: training fold 2 / 5
random_forest_no_rfe: training fold 3 / 5
random_forest_no_rfe: training fold 4 / 5
random_forest_no_rfe: training fold 5 / 5


In [46]:
rf_no_rfe_metrics = pd.DataFrame(columns=["fold", "features"] + metric_columns)

for i, fold in enumerate(no_rfe_folds):
    y_true = fold["y_val"]
    y_predi = rf_num_predictions[i]
    y_prob = rf_num_probabilities[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)

    rf_fold_ac = accuracy_score(y_true, y_predi)
    rf_fold_prec = precision_score(y_true, y_predi)
    rf_fold_re = recall_score(y_true, y_predi)
    rf_fold_f1 = f1_score(y_true, y_predi)
    rf_fold_roc = roc_auc_score(y_true, y_prob)
    rf_fold_prauc = auc(recall, precision)

    rf_fold_balanced_accuracy = balanced_accuracy_score(y_true, y_predi)
    rf_fold_specificity = recall_score(y_true, y_predi, pos_label=0)

    rf_no_rfe_metrics.loc[i] = [i + 1, fold["features"], rf_fold_ac, rf_fold_prec, rf_fold_re, rf_fold_f1, rf_fold_roc, rf_fold_prauc, rf_fold_balanced_accuracy, rf_fold_specificity]

rf_balanced_accuracy = rf_no_rfe_metrics["balanced_accuracy"].mean()
rf_specificity = rf_no_rfe_metrics["specificity"].mean()


In [47]:
rf_acc = rf_no_rfe_metrics["accuracy"].mean()
rf_precision = rf_no_rfe_metrics["precision"].mean()
rf_recall = rf_no_rfe_metrics["recall"].mean()
rf_f1 = rf_no_rfe_metrics["f1"].mean()
rf_roc_auc = rf_no_rfe_metrics["roc_auc"].mean()
rf_pr_auc = rf_no_rfe_metrics["pr_auc"].mean()
rf_no_rfe_summary = rf_no_rfe_metrics[metric_columns].agg(["mean", "std"]).T


In [48]:
display(rf_no_rfe_summary)


,mean,std
accuracy,0.760817,0.005654
precision,0.400277,0.007371
recall,0.624109,0.019842
f1,0.487611,0.007842
roc_auc,0.786005,0.017349
pr_auc,0.454185,0.039083
balanced_accuracy,0.707713,0.007400
specificity,0.791316,0.009272


In [49]:
display(rf_no_rfe_metrics)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,68.0,0.764967,0.406375,0.618182,0.490385,0.790716,0.413614,0.708005,0.797829
1,2.0,68.0,0.753880,0.389961,0.612121,0.476415,0.763447,0.419890,0.698869,0.785617
2,3.0,68.0,0.758047,0.400000,0.658537,0.497696,0.807095,0.509415,0.719363,0.780190
3,4.0,68.0,0.759156,0.396887,0.621951,0.484561,0.773828,0.459276,0.705820,0.789688
4,5.0,68.0,0.768036,0.408163,0.609756,0.488998,0.794937,0.468729,0.706506,0.803256


### Random forest: RFECV



In [50]:
rt_rfecv_models = []
rt_rfecv_predictions = []
rt_rfecv_probabilities = []

for fold_num, fold in enumerate(rfecv_folds, start=1):
    print("random_forest_rfecv: training fold", fold_num, "/", len(rfecv_folds))
    rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=2, max_features='log2', random_state=42)
    rf.fit(fold["x_train"], fold["y_train"])

    y_pred = rf.predict(fold["x_val"])
    y_prob = rf.predict_proba(fold["x_val"])[:, 1]
    rt_rfecv_predictions.append(y_pred)
    rt_rfecv_probabilities.append(y_prob)

    rt_rfecv_models.append({"fold": fold_num,"model": rf, "encoder": encoded_folds[fold_num - 1]["encoder"],
        "selector": fold["selector"],"input_columns": list(x_train.columns)})


random_forest_rfecv: training fold 1 / 5
random_forest_rfecv: training fold 2 / 5
random_forest_rfecv: training fold 3 / 5
random_forest_rfecv: training fold 4 / 5
random_forest_rfecv: training fold 5 / 5


In [51]:
rf_rfecv_metrics = pd.DataFrame(columns=["fold", "features"] + metric_columns)

for i, fold in enumerate(rfecv_folds):
    y_true = fold["y_val"]
    y_pred = rt_rfecv_predictions[i]
    y_prob = rt_rfecv_probabilities[i]
    precision, recall, _ = precision_recall_curve(y_true, y_prob, pos_label=1)
    rf_rfecv_fold_acc = accuracy_score(y_true, y_pred)
    rf_rfecv_fold_precision = precision_score(y_true, y_pred, zero_division=0)
    rf_rfecv_fold_recall = recall_score(y_true, y_pred, zero_division=0)
    rf_rfecv_fold_f1 = f1_score(y_true, y_pred, zero_division=0)
    rf_rfecv_fold_roc_auc = roc_auc_score(y_true, y_prob)
    rf_rfecv_fold_pr_auc = auc(recall, precision)

    rf_rfecv_fold_balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    rf_rfecv_fold_specificity = recall_score(y_true, y_pred, pos_label=0)

    rf_rfecv_metrics.loc[i] = [ i + 1, fold["features"], rf_rfecv_fold_acc, rf_rfecv_fold_precision, rf_rfecv_fold_recall, rf_rfecv_fold_f1, rf_rfecv_fold_roc_auc, rf_rfecv_fold_pr_auc, rf_rfecv_fold_balanced_accuracy, rf_rfecv_fold_specificity]

rf_rfecv_balanced_accuracy = rf_rfecv_metrics["balanced_accuracy"].mean()
rf_rfecv_specificity = rf_rfecv_metrics["specificity"].mean()


rf_rfecv_acc = rf_rfecv_metrics["accuracy"].mean()
rf_rfecv_precision = rf_rfecv_metrics["precision"].mean()
rf_rfecv_recall = rf_rfecv_metrics["recall"].mean()
rf_rfecv_f1 = rf_rfecv_metrics["f1"].mean()
rf_rfecv_roc_auc = rf_rfecv_metrics["roc_auc"].mean()
rf_rfecv_pr_auc = rf_rfecv_metrics["pr_auc"].mean()

rf_rfecv_summary = rf_rfecv_metrics[metric_columns].agg(["mean", "std"]).T


In [52]:
display(rf_rfecv_metrics)


,fold,features,accuracy,precision,recall,f1,roc_auc,pr_auc,balanced_accuracy,specificity
0,1.0,4.0,0.650776,0.323113,0.830303,0.465195,0.769204,0.453461,0.720443,0.610583
1,2.0,6.0,0.656319,0.317380,0.763636,0.448399,0.749007,0.438802,0.697965,0.632293
2,3.0,48.0,0.758047,0.403571,0.689024,0.509009,0.805730,0.509287,0.731215,0.773406
3,4.0,32.0,0.719201,0.352159,0.646341,0.455914,0.766071,0.455669,0.690878,0.735414
4,5.0,31.0,0.741398,0.383838,0.695122,0.494577,0.794573,0.462499,0.723409,0.751696


In [53]:
display(rf_rfecv_summary)


,mean,std
accuracy,0.705148,0.049118
precision,0.356013,0.037503
recall,0.724885,0.072380
f1,0.474619,0.026015
roc_auc,0.776917,0.022902
pr_auc,0.463944,0.026779
balanced_accuracy,0.712782,0.017398
specificity,0.700678,0.073980


Saving the 2 Random Forest classifier  models

In [54]:
joblib.dump({ "variant": "random_forest_no_rfe","positive_class": 1, "requires_mice_imputed_input": True,
    "fold_models": rf_num_models,}, MODEL_DIR / "random_forest_no_rfe.joblib")
print("saved random forest")

joblib.dump({ "variant": "random_forest_rfecv","positive_class": 1, "requires_mice_imputed_input": True,"fold_models": rt_rfecv_models,},
    MODEL_DIR / "random_forest_rfecv.joblib")

print("saved random forest model with RFEcv")
print("..............all the models are done saving...............")


saved random forest
saved random forest model with RFEcv
..............all the models are done saving...............


### Compare models ( RFEcv vs No RFE)


In [ ]:
comparison = pd.DataFrame({"variant": ["logistic_regression_no_rfe", "logistic_regression_rfecv", "gradient_boosting_no_rfe","gradient_boosting_rfecv","random_forest_no_rfe", "random_forest_rfecv",],
    "accuracy": [lr_acc, lr_rfecv_acc, gb_acc, gb_rfecv_acc, rf_acc, rf_rfecv_acc],
    "balanced_accuracy": [lr_balanced_accuracy, lr_rfecv_balanced_accuracy, gb_balanced_accuracy, gb_rfecv_balanced_accuracy, rf_balanced_accuracy, rf_rfecv_balanced_accuracy],
    "precision": [lr_precision, lr_rfecv_precision, gb_precision, gb_rfecv_precision, rf_precision, rf_rfecv_precision],
    "recall": [lr_recall, lr_rfecv_recall, gb_recall, gb_rfecv_recall, rf_recall, rf_rfecv_recall],
    "f1": [lr_f1, lr_rfecv_f1, gb_f1, gb_rfecv_f1, rf_f1, rf_rfecv_f1],
    "roc_auc": [lr_roc_auc, lr_rfecv_roc_auc, gb_roc_auc, gb_rfecv_roc_auc, rf_roc_auc, rf_rfecv_roc_auc],
    "pr_auc": [lr_pr_auc, lr_rfecv_pr_auc, gb_pr_auc, gb_rfecv_pr_auc, rf_pr_auc, rf_rfecv_pr_auc],
    "specificity": [lr_specificity, lr_rfecv_specificity, gb_specificity, gb_rfecv_specificity, rf_specificity, rf_rfecv_specificity],
}).set_index("variant")
display(comparison)
comparison.to_csv(OUTPUT / "six_model_comparison.csv")


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,specificity
variant,,,,,,,,
logistic_regression_no_rfe,0.715996,0.713364,0.359074,0.709224,0.476725,0.784928,0.464245,0.717503
logistic_regression_rfecv,0.720877,0.703189,0.358954,0.675307,0.468316,0.776327,0.472318,0.731072
gradient_boosting_no_rfe,0.808744,0.665680,0.473582,0.440451,0.456249,0.787095,0.462817,0.890909
gradient_boosting_rfecv,0.738221,0.691296,0.392647,0.617598,0.464333,0.774811,0.463137,0.764993
random_forest_no_rfe,0.760817,0.707713,0.400277,0.624109,0.487611,0.786005,0.454185,0.791316
random_forest_rfecv,0.705148,0.712782,0.356013,0.724885,0.474619,0.776917,0.463944,0.700678


In [56]:
display(comparison)


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,specificity
variant,,,,,,,,
logistic_regression_no_rfe,0.715996,0.713364,0.359074,0.709224,0.476725,0.784928,0.464245,0.717503
logistic_regression_rfecv,0.720877,0.703189,0.358954,0.675307,0.468316,0.776327,0.472318,0.731072
gradient_boosting_no_rfe,0.808744,0.665680,0.473582,0.440451,0.456249,0.787095,0.462817,0.890909
gradient_boosting_rfecv,0.738221,0.691296,0.392647,0.617598,0.464333,0.774811,0.463137,0.764993
random_forest_no_rfe,0.760817,0.707713,0.400277,0.624109,0.487611,0.786005,0.454185,0.791316
random_forest_rfecv,0.705148,0.712782,0.356013,0.724885,0.474619,0.776917,0.463944,0.700678


### Compare models with RFE 20 vs no RFE vs RFECV


In [57]:
lr_baseline = pd.read_csv("results/baseline/logistic_regression/cv_metrics.csv")
gb_baseline = pd.read_csv("results/baseline/gradient_boosting/cv_metrics.csv")
rf_baseline = pd.read_csv("results/baseline/random_forest/cv_metrics.csv")


baseline_rfe20 = pd.concat([lr_baseline, gb_baseline, rf_baseline])
baseline_rfe20 = baseline_rfe20[["Accuracy", "Precision", "Recall", "F1-score", "AUROC", "PR-AUC"]]
baseline_rfe20.columns = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
baseline_rfe20.index = ["logistic_regression_rfe20","gradient_boosting_rfe20","random_forest_rfe20"]
baseline_rfe20.index.name = "variant"


In [58]:
compare = pd.concat([baseline_rfe20, comparison[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]]])
compare = compare.loc[["logistic_regression_rfe20","logistic_regression_no_rfe", "logistic_regression_rfecv", "gradient_boosting_rfe20",
"gradient_boosting_no_rfe","gradient_boosting_rfecv", "random_forest_rfe20", "random_forest_no_rfe", "random_forest_rfecv"]]


compare["accuracy_difference_pp"] = 0.0
compare["recall_difference_pp"] = 0.0

for model_name in ["logistic_regression", "gradient_boosting", "random_forest"]:
    baseline_accuracy = compare.loc[model_name + "_rfe20", "accuracy"]
    baseline_recall = compare.loc[model_name + "_rfe20", "recall"]

    for method in ["no_rfe", "rfecv"]:
        row = model_name + "_" + method
        compare.loc[row, "accuracy_difference_pp"] = (compare.loc[row, "accuracy"] - baseline_accuracy) * 100
        compare.loc[row, "recall_difference_pp"] = (compare.loc[row, "recall"] - baseline_recall) * 100



#### Logistic Regression


In [ ]:
percentage_columns = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
compare_percent = compare.copy()
compare_percent[percentage_columns] = compare_percent[percentage_columns] * 100
compare_percent.to_csv(OUTPUT / "Experiment1_comparison.csv")

lr_comparison = compare_percent.loc[["logistic_regression_rfe20","logistic_regression_no_rfe","logistic_regression_rfecv"]].copy()
lr_comparison.index = ["RFE 20", "No RFE", "RFECV"]
lr_comparison.index.name = "Method"

display(lr_comparison.style
    .format("{:.2f}%", subset=percentage_columns)
    .format("{:+.2f} pp", subset=["accuracy_difference_pp", "recall_difference_pp"]))
lr_comparison.to_csv(OUTPUT / "logistic_regression_comparison_percent.csv")


,accuracy,precision,recall,f1,roc_auc,pr_auc,accuracy_difference_pp,recall_difference_pp
Method,,,,,,,,
RFE 20,71.64%,36.27%,73.12%,48.48%,78.40%,46.01%,+0.00 pp,+0.00 pp
No RFE,71.60%,35.91%,70.92%,47.67%,78.49%,46.42%,-0.04 pp,-2.19 pp
RFECV,72.09%,35.90%,67.53%,46.83%,77.63%,47.23%,+0.44 pp,-5.58 pp


#### Gradient Boosting


In [ ]:
gb_comparison = compare_percent.loc[["gradient_boosting_rfe20", "gradient_boosting_no_rfe", "gradient_boosting_rfecv"]].copy()
gb_comparison.index = ["RFE 20", "No RFE", "RFECV"]
gb_comparison.index.name = "Method"

display( gb_comparison.style
    .format("{:.2f}%", subset=percentage_columns)
    .format("{:+.2f} pp", subset=["accuracy_difference_pp", "recall_difference_pp"]))
gb_comparison.to_csv(OUTPUT / "gradient_boosting_comparison_percent.csv")


,accuracy,precision,recall,f1,roc_auc,pr_auc,accuracy_difference_pp,recall_difference_pp
Method,,,,,,,,
RFE 20,74.44%,38.03%,63.14%,47.44%,77.42%,44.84%,+0.00 pp,+0.00 pp
No RFE,80.87%,47.36%,44.05%,45.62%,78.71%,46.28%,+6.43 pp,-19.10 pp
RFECV,73.82%,39.26%,61.76%,46.43%,77.48%,46.31%,-0.62 pp,-1.38 pp


#### Random Forest


In [ ]:
rf_comparison = compare_percent.loc[[ "random_forest_rfe20","random_forest_no_rfe","random_forest_rfecv"]].copy()
rf_comparison.index = ["RFE 20", "No RFE", "RFECV"]
rf_comparison.index.name = "Method"

display(rf_comparison.style
    .format("{:.2f}%", subset=percentage_columns)
    .format("{:+.2f} pp", subset=["accuracy_difference_pp", "recall_difference_pp"]))
rf_comparison.to_csv(OUTPUT / "random_forest_comparison_percent.csv")


,accuracy,precision,recall,f1,roc_auc,pr_auc,accuracy_difference_pp,recall_difference_pp
Method,,,,,,,,
RFE 20,72.27%,36.49%,70.08%,47.99%,77.88%,44.77%,+0.00 pp,+0.00 pp
No RFE,76.08%,40.03%,62.41%,48.76%,78.60%,45.42%,+3.82 pp,-7.66 pp
RFECV,70.51%,35.60%,72.49%,47.46%,77.69%,46.39%,-1.75 pp,+2.41 pp


## Model Evaluation

Based on the validation results it is harder to say which method is best and weak. Because the
way the results are distributed changes for each model. For Logistic regression RFE20 achieved
the best recall and F1 score but had lower accuracy than RFECV. In Gradient Boosting its
different. Without RFE was able to achieved the highest accuracy (80.87%) but its recall and F1
score were lower compared to the other two methods. Same in Random Forest model. It
performed well without RFE reaching the highest accuracy, precision and F1 score, but lower
recall compared to the other two methods. These results show the reason why my criticized about
authors model selection, only based on accuracy instead of other metrics
